In [ ]:
#@title 按這裡開始（先按 ▶）
print('✅ W14 出發！本週目標：用加速度計資料教電腦分辨「走路」和「靜止」')
RAW = 'https://raw.githubusercontent.com/myliao2007/stust-course-1151/main/ai-intro-mobile/data/'
import pandas as pd, numpy as np   # 先把資料工具帶上
print('先用 phyphox 錄好再來；還沒錄也沒關係，全程可以用示範資料跑完')

# W14 手機感測器：活動辨識
手機開這本請先切成「電腦版網站」（iPhone：網址列 ᴀA → 要求電腦版網站；Android：右上 ⋮ → 電腦版網站），再點每一格左邊的 ▶ 執行。

用 phyphox 的 Acceleration (without g) 錄每種活動 30～60 秒，Export Data 選 CSV，檔名改成活動名（如 walk.csv、still.csv）。沒有檔案就直接用示範資料。

**任務三之一**：上傳 phyphox 匯出的 CSV（可一次選多個）。沒有檔案就按「取消」，會自動用示範資料。

In [ ]:
#@title 任務三之一：上傳 phyphox 的 CSV（沒有就按取消）
from google.colab import files
try:
    up = files.upload()      # 檔名請先改成活動名，例如 walk.csv
except Exception:
    up = {}
print('✅ 收到', len(up), '個檔案；0 個也沒關係，下一格會自動用示範資料')

**任務三之二**：整理成「活動 → 資料表」。phyphox 的欄位是 Time (s) 與 Acceleration x/y/z (m/s^2)，這裡把三軸合成總加速度 a。

In [ ]:
#@title 任務三之二：整理資料（讀不到就退回示範資料）
up = globals().get('up', {})     # ←沒跑上傳格就當作沒有檔案
def load(src):   # phyphox 欄位：Time (s), Acceleration x/y/z (m/s^2)
    t = pd.read_csv(src).iloc[:, :4]
    t.columns = ['t', 'x', 'y', 'z']
    t['a'] = np.sqrt(t.x**2 + t.y**2 + t.z**2)
    return t
acts = {n.split('.')[0]: load(n) for n in up}
if not acts:   # 示範資料：前 60 秒靜止、後 60 秒走路（50Hz 模擬）
    d = load(RAW + 'demo_motion.csv')
    acts = {'still': d[d.t < 60], 'walk': d[d.t >= 60]}
print('✅ 活動清單：', list(acts), '→ 往下畫波形')

**任務三之三**：畫波形。走路的波形上下震、靜止是一條平線——眼睛先看得出差別，電腦才學得會。

In [ ]:
#@title 任務三之三：畫出每種活動的波形
import matplotlib.pyplot as plt
n = len(acts)
fig, ax = plt.subplots(n, 1, figsize=(8, 2 * n))
for i, (k, t) in enumerate(acts.items()):
    a = ax[i] if n > 1 else ax
    a.plot(t['t'], t['a'], lw=0.5)
    a.set_title(k)
plt.tight_layout()
plt.show()
print('✅ 四處亂震的是動、平線是靜 → 往下切窗訓練')

**任務四**：切窗算特徵並訓練。win 是每個視窗幾筆（50Hz 時 100 筆＝2 秒）。win 太小雜訊多，太大樣本少，試 50、100、200 各跑一次。

In [ ]:
#@title 任務四：切窗算特徵並訓練（改 win 比較）
win = 100  #@param {type:"integer"}
def feat(t, y):
    g = t['a'].groupby(t.index // win)
    f = pd.DataFrame({'m': g.mean(), 's': g.std()})
    f['y'] = y
    return f.dropna()
data = pd.concat([feat(t, k) for k, t in acts.items()])
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
sc = cross_val_score(KNeighborsClassifier(3), data[['m', 's']], data['y'], cv=3)
print('✅ 各活動平均正確率 =', round(sc.mean(), 3), '→ 改 win 再跑一次比較')

**延伸：規則法**。不用機器學習，一條「震動夠大就是走路」的規則能拿幾分？

In [ ]:
#@title 延伸：一條規則就能分？（改門檻 th 試試）
th = 0.3  #@param {type:"number"}

guess = np.where(data['s'] > th, 'walk', 'still')
acc = (guess == data['y']).mean()
print(f'規則「視窗std > {th} 就算走路」正確率 = {acc:.2f}')
print('✅ 調 th 比比看：規則法贏得了 KNN 嗎？')

In [ ]:
#@title 收工檢查（直接按 ▶）
print('本週要交：波形圖截圖＋三個 win 的正確率＋規則法 vs KNN 的一句話結論')
print('✅ 上傳課程表單，檔名：AI導論_W14_學號_姓名')